# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RodionOm/Search-ranking-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The rule in plain words:** a page is worth reviewing first if, *before* the label window, it already had real search demand, it is still young, people saw it but rarely clicked, its impressions had just jumped, and it targets a real keyword. Each condition that holds adds one point; ties are broken by prev-window impressions.

**Leakage-safe by design.** The label compares the last 30 days with the prev 30 days, so the rule only reads data from *before* the label window (see ML-05):

```
 day -90           day -60           day -30           export
    |   older 30d     |    prev 30d     |    last 30d     |
                                        ^ moment of prediction
```

| Reason code | Condition | Why it should matter |
|---|---|---|
| `visible_page` | prev-30d impressions ≥ 200 | only pages people actually see are worth a refresh |
| `young_page` | content age ≤ 180 days | new pages often lose their launch boost |
| `low_ctr_page` | prev-30d CTR < 0.1% and impressions ≥ 200 | seen but not clicked: title/snippet problem |
| `recent_spike` | prev-30d impressions ≥ ~1.35× older-30d | after a spike, traffic often falls back |
| `keyword_target` | page has target-keyword data | pages without a keyword target decline far less often |

The cell below measures how often each reason fires and how often it is right, against the base rate.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    !git clone -q https://github.com/flyrank-bih/flyrank-ml-internship-starter.git
    %cd flyrank-ml-internship-starter
print(os.getcwd())
import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# only data from BEFORE the label window: prev = days 31-60, older = days 61-90
prev  = df["impressions_prev_30d"]
older = df["impressions_90d"] - df["impressions_last_30d"] - df["impressions_prev_30d"]
ctr_prev = np.where(prev > 0, df["clicks_prev_30d"] / prev.replace(0, 1) * 100, 0)
momentum = np.log1p(prev) - np.log1p(older)

flags = pd.DataFrame(index=df.index)
flags["visible_page"]   = prev >= 200
flags["young_page"]     = df["content_age_days"] <= 180
flags["low_ctr_page"]   = (ctr_prev < 0.1) & (prev >= 200)
flags["recent_spike"]   = momentum > 0.3
flags["keyword_target"] = df["search_volume"].notna()

# how often each reason fires, and how often it is right
summary = pd.DataFrame({
    "pages_flagged": flags.sum(),
    "share_declining": [df.loc[flags[c], "is_declining_label"].mean() for c in flags.columns],
}).round(3)
print(f"base rate: {df['is_declining_label'].mean():.3f}\n")
print(summary)

/content/flyrank-ml-internship-starter
base rate: 0.542

                pages_flagged  share_declining
visible_page            15212            0.615
young_page              12272            0.627
low_ctr_page             6404            0.690
recent_spike             5545            0.605
keyword_target          27532            0.564


## 2. Build the ranked queue (writes the CSV)

Score = number of reasons that fire (0–5) + a tie-break in [0, 0.5) from prev-30d impressions, so a page with more reasons always ranks above a page with fewer. Every page is ranked, written to `work/outputs/baseline_action_score.csv`, and evaluated with precision@K next to the base rate and a random-ranking floor.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
tiebreak = np.log1p(prev).rank(pct=True) * 0.5          # 0..0.5, never beats a whole point
df["baseline_score"] = flags.sum(axis=1) + tiebreak
df["reason_codes"]   = flags.apply(lambda row: "|".join(row.index[row]) or "none", axis=1)
df["baseline_rank"]  = df["baseline_score"].rank(method="first", ascending=False).astype(int)

queue = df.sort_values("baseline_rank")
os.makedirs("work/outputs", exist_ok=True)
queue[["baseline_rank", "content_id", "client_id", "baseline_score", "reason_codes",
       "impressions_prev_30d", "content_age_days"]].to_csv("work/outputs/baseline_action_score.csv", index=False)

base = df["is_declining_label"].mean()
print(f"base rate: {base:.3f}")
for k in [10, 20, 50, 500]:
    p = queue["is_declining_label"].head(k).mean()
    print(f"P@{k:<4} {p:.2f}   lift {p / base:.2f}x")

# floor: random ranking, averaged over 200 shuffles
rand50 = np.mean([df["is_declining_label"].sample(frac=1, random_state=s).head(50).mean() for s in range(200)])
print(f"random P@50 (200 shuffles): {rand50:.2f}")
print("rows written:", len(queue))

base rate: 0.542
P@10   0.60   lift 1.11x
P@20   0.65   lift 1.20x
P@50   0.80   lift 1.48x
P@500  0.78   lift 1.43x
random P@50 (200 shuffles): 0.54
rows written: 30000


## 3. Top-20 review

The table below comes from the code cell under it. `trend_direction` is shown **only to check the outcome after the review** — it is not used by the score. Short client tags are pseudonym suffixes, for reading only.

All 20 pages fire all 5 reasons, so the order inside the top 20 is set by the tie-break (prev-window impressions). The shared symptom is a very low prev-window CTR (≤ 0.10%) on large demand, so the first action is always on the search snippet.

| Rank | Action | Main reason | Confidence | What would make it wrong |
|---|---|---|---|---|
| 1 | rewrite title + meta, check query match | low CTR at 97k impressions, 3.1× spike | medium | the spike was a breakout that holds at the new level |
| 2 | rewrite title + meta, then refresh content | low CTR, oldest page in top 20 (154 d) | high (small 1.4× spike, steady demand) | impressions come from queries the page does not answer |
| 3 | rewrite title + meta | CTR 0.02% at 85k impressions | medium | breakout that holds; 3 of the top 6 share this client |
| 4 | rewrite title + meta, then refresh | CTR 0.03%, 2.2× spike | medium | same-client batch effect, one cause for many pages |
| 5 | rewrite title + meta | 6.6× spike, CTR 0.01% | medium | a one-off news spike that was never search demand |
| 6 | rewrite title + meta | CTR 0.02% | medium | same client as ranks 1, 3, 5 — one site-level cause |
| 7 | rewrite title + meta | CTR 0.07%, mild 1.6× spike | low | mild spike band is the weakest signal (section 4) |
| 8 | rewrite title + meta, then refresh | 10.3× spike, CTR 0.04% | medium | huge spike may be a ranking breakout, not a blip |
| 9 | rewrite title + meta | 5.8× spike, CTR at the 0.10% edge | medium | CTR is right at the threshold — rule is fragile here |
| 10 | rewrite title + meta | CTR 0.09%, 1.9× spike | low | mild spike band; CTR near threshold |
| 11 | rewrite title + meta | CTR 0.02%, 2.3× spike | medium | query mismatch rather than a stale snippet |
| 12 | rewrite title + meta, then refresh | CTR 0.07%, 153 d old | medium | same-client batch effect |
| 13 | rewrite title + meta | CTR 0.00% at 24k impressions | medium | 0 clicks can mean a tracking gap, not a bad snippet |
| 14 | rewrite title + meta | 3.5× spike, CTR 0.03% | medium | same-client batch effect |
| 15 | rewrite title + meta | 4.2× spike, CTR 0.05% | medium | same-client batch effect |
| 16 | rewrite title + meta | CTR 0.09%, small 1.4× spike | low | weakest spike and CTR near threshold |
| 17 | monitor first, then title + meta | 54.7× spike from 394 impressions | low | the page just started ranking — a breakout, not a decline risk |
| 18 | rewrite title + meta | CTR at the 0.10% edge, 1.7× spike | low | threshold edge + mild spike |
| 19 | rewrite title + meta | CTR 0.06%, 1.8× spike | medium | query mismatch |
| 20 | rewrite title + meta, then refresh | CTR 0.07% | medium | same-client batch effect |

**Top-10 (portal card):** P@10 = 0.60 (6 of 10 declined). The 4 misses (ranks 1, 3, 7, 10) all came from the same two patterns analysed in section 4.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top = queue.head(20).copy()
top["older_30d"]  = (top["impressions_90d"] - top["impressions_last_30d"] - top["impressions_prev_30d"])
top["spike_x"]    = (top["impressions_prev_30d"] / top["older_30d"].replace(0, 1)).round(1)
top["ctr_prev_%"] = (top["clicks_prev_30d"] / top["impressions_prev_30d"] * 100).round(2)
top["client"]     = top["client_id"].str[-4:]          # short pseudonym, for reading only

cols = ["baseline_rank", "client", "older_30d", "impressions_prev_30d", "spike_x",
        "ctr_prev_%", "content_age_days", "trend_direction"]
print(top[cols].to_string(index=False))
print("\nclients in top 20:", top["client"].nunique())
print(top["client"].value_counts().to_dict())

 baseline_rank client  older_30d  impressions_prev_30d  spike_x  ctr_prev_%  content_age_days trend_direction
             1   89fc      30878                 97200      3.1        0.09                97          stable
             2   201b      67433                 96122      1.4        0.04               154            down
             3   89fc      39161                 84773      2.2        0.02                95          stable
             4   27de      29469                 65928      2.2        0.03               153            down
             5   89fc       9840                 64917      6.6        0.01                97            down
             6   89fc      31766                 54506      1.7        0.02                95            down
             7   d7e2      32254                 53104      1.6        0.07               147          stable
             8   27de       4898                 50269     10.3        0.04               148            down
          

## 4. Weak picks + leakage check

**Weak picks.** 7 of the top 20 did not decline (6 stable, 1 up). Two patterns explain them:

1. **The spike flag is the weakest reason.** Among visible pages, a mild spike (1.35–3×) declines *less* often (0.548) than no spike at all (0.625). 6 of the 7 misses sit in that band. A mild jump often means the page climbed to a new level and stayed there, not a blip that falls back.
2. **Breakouts look like spikes.** Rank 17 went from 394 to 21,552 impressions (54.7×): a page that just started ranking, not one about to lose traffic. The rule cannot tell a breakout from a blip.

**Concentration.** 13 of the top 20 come from two clients (7 + 6). Same-client pages often share one cause (a site-level change), so the top 20 is fewer independent problems than it looks. Later evaluation uses client-grouped splits.

I keep the baseline **frozen** as the bar the model must beat; the spike finding is a hypothesis for ML-08, not a reason to tune the rule after looking at the answers.

**Leakage check.**
- The score reads only: prev-30d impressions and clicks, content age, keyword presence, and the older-30d window.
- `impressions_90d` and `impressions_last_30d` are read **only to subtract out** the older window (`older = 90d − last − prev`). The check confirms older is never negative and that older + prev + last rebuilds the 90d total exactly, so no last-30d information survives in the score.
- No product flags (`provider_used`, `model_used`), no tiers, no 90-day rates (`ctr`, `avg_position`), no `days_since_last_update`, no label columns.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1) weak picks: top-20 pages that did NOT decline
weak = top[top["is_declining_label"] == 0]
print("weak picks in top 20:", len(weak))
print(weak[["baseline_rank", "client", "older_30d", "impressions_prev_30d", "spike_x", "trend_direction"]].to_string(index=False))

# 2) is the spike flag the culprit? share declining by spike size among visible pages
vis = df[flags["visible_page"]].copy()
vis["spike_x"] = (prev / older.replace(0, 1))[flags["visible_page"]]
vis["spike_band"] = pd.cut(vis["spike_x"], [0, 1.35, 3, 10, np.inf], labels=["<1.35x", "1.35-3x", "3-10x", ">10x"])
print("\nshare declining by spike size (visible pages):")
print(vis.groupby("spike_band", observed=True)["is_declining_label"].agg(["count", "mean"]).round(3))

# 3) leakage check: which raw columns does the score read?
score_inputs = ["impressions_prev_30d", "clicks_prev_30d", "impressions_90d", "impressions_last_30d",
                "content_age_days", "search_volume"]
banned = ["trend_direction", "trend_pct", "ctr", "avg_position", "days_since_last_update",
          "provider_used", "model_used", "impression_tier", "position_tier", "freshness_tier"]
print("\nbanned columns used by the score:", [c for c in banned if c in score_inputs] or "none")

# impressions_90d and impressions_last_30d are read ONLY to subtract out the older window
print("older window never negative:", bool((older >= 0).all()))
print("older + prev + last == 90d:", bool(((older + prev + df["impressions_last_30d"]) == df["impressions_90d"]).all()))

weak picks in top 20: 7
 baseline_rank client  older_30d  impressions_prev_30d  spike_x trend_direction
             1   89fc      30878                 97200      3.1          stable
             3   89fc      39161                 84773      2.2          stable
             7   d7e2      32254                 53104      1.6          stable
            10   89fc      16504                 30774      1.9          stable
            16   0f77      15198                 21773      1.4              up
            17   5ae5        394                 21552     54.7          stable
            18   bef1      12460                 21049      1.7          stable

share declining by spike size (visible pages):
            count   mean
spike_band              
<1.35x      12493  0.625
1.35-3x      2067  0.548
3-10x         494  0.652
>10x          158  0.608

banned columns used by the score: none
older window never negative: True
older + prev + last == 90d: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.